# Stage 2 Real ResNet-Trigger Report Notebook

This notebook prepares paper-facing plots and summary tables for the Stage 2 real ResNet-trigger campaign. It reads the generated CSVs, the append-only JSONL ledger, and the captured run log rather than hard-coding metrics.

Primary campaign: `resnet_real_incremental`.

In [ ]:
from pathlib import Path
import ast
import csv
import json
import re

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

CAMPAIGN_ID = 'resnet_real_incremental'
ledger_path = ROOT / 'experiments' / 'ledgers' / f'{CAMPAIGN_ID}_trials.jsonl'
summary_path = ROOT / 'paper' / 'tables' / 'main_campaign_summary.csv'
governance_path = ROOT / 'paper' / 'tables' / 'governance_metrics.csv'
trajectory_path = ROOT / 'paper' / 'figures' / 'campaign_trajectory.csv'
decision_counts_path = ROOT / 'paper' / 'figures' / 'accepted_discarded_invalid_counts.csv'
artifact_dir = ROOT / 'experiments' / 'artifacts' / CAMPAIGN_ID / 'trial-001'
run_log_path = artifact_dir / 'run.log'
patch_path = artifact_dir / 'patch.diff'

def read_csv(path):
    with path.open(newline='') as f:
        return list(csv.DictReader(f))

records = [json.loads(line) for line in ledger_path.read_text().splitlines() if line.strip()]
summary = read_csv(summary_path)
governance = read_csv(governance_path)
trajectory = read_csv(trajectory_path)
decision_counts = read_csv(decision_counts_path)

print('records:', len(records))
print('campaign:', records[0]['campaign_id'])
print('decision:', records[0]['decision'])
print('metric:', records[0]['parsed_metrics'])

## Campaign Summary

The summary and governance tables are the paper-facing outputs exported by the Stage 2 reporting pipeline.

In [ ]:
summary[0], governance[0]

## Parse Epoch-Level Metrics From The Real Run Log

In [ ]:
log_text = run_log_path.read_text(encoding='utf-8', errors='replace')
epoch_rows = []
for match in re.finditer(r"^\{'epoch':.*\}$", log_text, flags=re.MULTILINE):
    epoch_rows.append(ast.literal_eval(match.group(0)))

final_metrics = {}
for key in ['val_bpb', 'training_seconds', 'total_seconds', 'num_steps', 'num_params_M', 'epochs_completed', 'val_auc', 'val_pr_auc', 'val_roc_auc']:
    m = re.search(rf'^{re.escape(key)}:\s+([-+]?\d+(?:\.\d+)?)', log_text, flags=re.MULTILINE)
    if m:
        final_metrics[key] = float(m.group(1))

print('epoch rows:')
for row in epoch_rows:
    print(row)
print('\nfinal metrics:', final_metrics)

## Plot Helpers

The plotting code uses only the Python standard library and writes SVG files, so it works even when `matplotlib` is not installed.

In [ ]:
FIG_DIR = ROOT / 'paper' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

def line_plot_svg(points, path, title, x_label, y_label, y_ticks=None, width=760, height=420):
    ml, mr, mt, mb = 80, 30, 40, 65
    plot_w, plot_h = width - ml - mr, height - mt - mb
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    xmin, xmax = min(xs), max(xs)
    ymin, ymax = min(ys), max(ys)
    pad = max((ymax - ymin) * 0.12, 0.001)
    ymin -= pad
    ymax += pad
    def px(x):
        return ml + (x - xmin) / (xmax - xmin) * plot_w if xmax != xmin else ml + plot_w / 2
    def py(y):
        return mt + (ymax - y) / (ymax - ymin) * plot_h if ymax != ymin else mt + plot_h / 2
    poly = ' '.join(f'{px(x):.1f},{py(y):.1f}' for x, y in points)
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">']
    parts += ['<rect width="100%" height="100%" fill="white"/>', f'<text x="{ml}" y="26" font-family="Arial" font-size="18" font-weight="700">{title}</text>']
    parts += [f'<line x1="{ml}" y1="{mt+plot_h}" x2="{ml+plot_w}" y2="{mt+plot_h}" stroke="#333"/>', f'<line x1="{ml}" y1="{mt}" x2="{ml}" y2="{mt+plot_h}" stroke="#333"/>']
    for y in (y_ticks or ys):
        yy = py(y)
        parts += [f'<line x1="{ml}" y1="{yy:.1f}" x2="{ml+plot_w}" y2="{yy:.1f}" stroke="#e5e7eb"/>', f'<text x="{ml-8}" y="{yy+4:.1f}" text-anchor="end" font-family="Arial" font-size="11">{y:.3f}</text>']
    parts.append(f'<polyline fill="none" stroke="#2563eb" stroke-width="3" points="{poly}"/>')
    for x, y in points:
        parts.append(f'<circle cx="{px(x):.1f}" cy="{py(y):.1f}" r="4" fill="#2563eb"/>')
        parts.append(f'<text x="{px(x):.1f}" y="{py(y)-10:.1f}" text-anchor="middle" font-family="Arial" font-size="11">{y:.6f}</text>')
    parts += [f'<text x="{ml+plot_w/2}" y="{height-18}" text-anchor="middle" font-family="Arial" font-size="13">{x_label}</text>', f'<text x="22" y="{mt+plot_h/2}" transform="rotate(-90 22 {mt+plot_h/2})" text-anchor="middle" font-family="Arial" font-size="13">{y_label}</text>', '</svg>']
    path.write_text('\n'.join(parts) + '\n', encoding='utf-8')
    return path

def bar_plot_svg(items, path, title, width=680, height=380):
    ml, mt = 90, 55
    plot_w, plot_h = 520, 240
    maxv = max(v for _, v, _ in items) or 1
    parts = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">', '<rect width="100%" height="100%" fill="white"/>', f'<text x="{ml}" y="30" font-family="Arial" font-size="18" font-weight="700">{title}</text>', f'<line x1="{ml}" y1="{mt+plot_h}" x2="{ml+plot_w}" y2="{mt+plot_h}" stroke="#333"/>', f'<line x1="{ml}" y1="{mt}" x2="{ml}" y2="{mt+plot_h}" stroke="#333"/>']
    step = plot_w / len(items)
    for i, (label, value, color) in enumerate(items):
        x = ml + i * step + step * 0.2
        bar_w = step * 0.6
        bar_h = (value / maxv) * (plot_h - 10)
        y = mt + plot_h - bar_h
        parts += [f'<rect x="{x:.1f}" y="{y:.1f}" width="{bar_w:.1f}" height="{bar_h:.1f}" fill="{color}"/>', f'<text x="{x+bar_w/2:.1f}" y="{max(y-8, 45):.1f}" text-anchor="middle" font-family="Arial" font-size="13" font-weight="700">{value}</text>', f'<text x="{x+bar_w/2:.1f}" y="{mt+plot_h+25}" text-anchor="middle" font-family="Arial" font-size="12">{label}</text>']
    parts.append('</svg>')
    path.write_text('\n'.join(parts) + '\n', encoding='utf-8')
    return path

## Campaign Trajectory And Decision Plots

In [ ]:
campaign_points = [(idx, float(r['metric'])) for idx, r in enumerate(trajectory)]
decision_items = []
colors = {'kept': '#16a34a', 'discarded': '#f59e0b', 'failed_invalid': '#dc2626'}
for row in decision_counts:
    decision_items.append((row['decision'], int(row['count']), colors.get(row['decision'], '#64748b')))

campaign_svg = line_plot_svg(
    campaign_points,
    FIG_DIR / f'{CAMPAIGN_ID}_trajectory.svg',
    'Stage 2 Real ResNet Campaign Trajectory',
    'Measurement step (0=baseline, 1=agent trial)',
    'val_auc',
)
decision_svg = bar_plot_svg(
    decision_items,
    FIG_DIR / f'{CAMPAIGN_ID}_decisions.svg',
    'Stage 2 Real ResNet Decisions',
)
campaign_svg, decision_svg

## Epoch-Level Validation AUC Plot

In [ ]:
epoch_points = [(int(r['epoch']), float(r['val_auc'])) for r in epoch_rows]
epoch_svg = line_plot_svg(
    epoch_points,
    FIG_DIR / f'{CAMPAIGN_ID}_epoch_val_auc.svg',
    'Real ResNet Trial Epoch Validation AUC',
    'Epoch',
    'val_auc',
)
epoch_svg

## Patch And Artifact Audit

In [ ]:
print(patch_path.read_text())
print('Artifacts:')
for path in sorted(artifact_dir.iterdir()):
    print('-', path.relative_to(ROOT))

## Takeaways

- Stage 2 executed a real ResNet-trigger trial through the governed control plane.
- The worker produced a bounded `train.py` hyperparameter edit: `LEARNING_RATE = 1e-3` to `5e-4`.
- The trial was valid, accepted as the first valid metric, and fully auditable through the JSONL ledger and artifact directory.
- Governance metrics report no invalid edit-scope violation, no command failure, no metric parsing failure, complete provenance, and complete artifact capture.
- This is a real-run demonstration of the Stage 2 evaluation surface; larger multi-trial campaigns are still needed for optimization claims.